In [20]:
import pandas as pd
df = pd.read_parquet('../../data/olist/datos_requeridos/olist_final.parquet')
df.head(5)

,order_id,customer_id,order_purchase_timestamp,product_id,precio_items,precio_total
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,[87285b34884572647811a353c7ac498a],29.99,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,[595fac2a385ac33a80bd5114aec74eb8],118.70,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,[aa4383b373c6aca5d8797843e5594415],159.90,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,[d0b61bfb1de832b15ba9d266ca96e5b0],45.00,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,[65266b2da20d04dbe00c5c2d3bb7859e],19.90,28.62


In [21]:
df = df.assign(
    order_id=lambda x: x.order_id.astype("string"),
    customer_id=lambda x: x.customer_id.astype("string"),
    order_purchase_timestamp=lambda x: pd.to_datetime(x.order_purchase_timestamp)
)
df.dtypes


order_id                    string[python]
customer_id                 string[python]
order_purchase_timestamp    datetime64[ns]
product_id                          object
precio_items                       float64
precio_total                       float64
dtype: object

## VERIFICAR

In [33]:
hay_compras = df["customer_id"].value_counts()
hay_compras

customer_id
edb027a75a1449115f6b43211ae02a24    1
9ef432eb6251297304e76186b10a928d    1
b0830fb4747a6c6d20dea0b8c802d7ef    1
41ce2a54c0b03bf3443c3d931a367089    1
f88197465ea7920adcdbec7375364d82    1
                                   ..
3df704f53d3f1d4818840b34ec672a9f    1
19402a48fe860416adf93348aba37740    1
d3e3b74c766bc6214e0c830b17ee2341    1
7711cf624183d843aafe81855097bc37    1
494dded5b201313c64ed7f100595b95c    1
Name: count, Length: 98666, dtype: Int64

# **L**

Primero hay que definir el intervalo que se quiere evaluar (de momento voy a hacerlo con el timestamp completo, aunque va a haber que simplificarlo fijo)

In [27]:
periodo_inicial = '2017-01-01 00:00:00'  # fecha de inicio del periodo que queremos analizar
periodo_final   = '2025-12-31 00:00:00'  # fecha de fin del periodo que queremos analizar


In [28]:
df_periodo = df[
    (df['order_purchase_timestamp'] >= pd.Timestamp(periodo_inicial)) &
    (df['order_purchase_timestamp'] < pd.Timestamp(periodo_final))
]


In [30]:
# 1) Calcular min y max timestamp por cliente dentro del periodo
L_por_cliente = (
    df_periodo
    .groupby("customer_id", as_index=False)
    .agg(
        first_ts=("order_purchase_timestamp", "min"),
        last_ts=("order_purchase_timestamp", "max"),
        n_transacciones=("order_id", "nunique"),
    )
)

# 2) Calcular L (Length) como diferencia temporal
#    - en días (float) para no perder decimales si quieres
L_por_cliente["L_dias"] = (
    (L_por_cliente["last_ts"] - L_por_cliente["first_ts"])
    .dt.total_seconds() / 86400
)

# (opcional) si prefieres entero de días:
# L_por_cliente["L_dias"] = (L_por_cliente["last_ts"] - L_por_cliente["first_ts"]).dt.days

L_por_cliente.sort_values("L_dias").head(10)


,customer_id,first_ts,last_ts,n_transacciones,L_dias
98351,fffeda5b6d849fbd39689bb92087f431,2018-05-22 13:36:02,2018-05-22 13:36:02,1,0.0
98350,fffecc9f79fd8c764f843e9951b11341,2018-03-29 16:59:26,2018-03-29 16:59:26,1,0.0
98349,fffcb937e9dd47a13f05ecb8290f4d3e,2018-03-17 00:55:27,2018-03-17 00:55:27,1,0.0
98348,fffb97495f78be80e2759335275df2aa,2018-01-16 14:51:35,2018-01-16 14:51:35,1,0.0
98347,fffa0238b217e18a8adeeda0669923a3,2017-09-11 16:02:18,2017-09-11 16:02:18,1,0.0
98346,fff93c1da78dafaaa304ff032abc6205,2018-06-13 01:57:22,2018-06-13 01:57:22,1,0.0
98345,fff906ecb75de5809be384e0f8d65e45,2018-03-12 23:42:22,2018-03-12 23:42:22,1,0.0
98344,fff89c8ed4fcf69a823c1d149e429a0b,2017-12-16 12:53:03,2017-12-16 12:53:03,1,0.0
98343,fff7466a253c0e59499ea943462c10f9,2018-02-14 13:22:12,2018-02-14 13:22:12,1,0.0
98342,fff675a0d5924b9162b4a1bf410466cd,2017-12-06 12:51:27,2017-12-06 12:51:27,1,0.0
